In [168]:
#!pip install langchain

In [169]:
#!pip install langchain_openai

In [170]:
#!pip install langchain-core requests

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [5]:
from getpass import getpass
OPENAI_API_KEY=getpass("Enter OPENAI API key")

Enter OPENAI API key··········


In [6]:
import os
os.environ["OPENAI_API_KEY"]=OPENAI_API_KEY

In [7]:
llm=ChatOpenAI(model="gpt-4o-mini")

In [9]:
from getpass import getpass
EXCHANGE_RATE_API_KEY=getpass("Enter Exchange Rate API key")

Enter Exchange Rate API key··········


In [8]:
# tool create

In [125]:
@tool
def get_conversion_factor(base_currency:str,target_currency:str)->float:
  """Returns ONLY the conversion rate between two currencies."""

  url_domain="https://v6.exchangerate-api.com/v6/"
  url_end_point=f"/pair/{base_currency}/{target_currency}"
  final_url=f"{url_domain}{EXCHANGE_RATE_API_KEY}{url_end_point}"
  response=requests.get(final_url)
  return response.json()


In [126]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1779840001,
 'time_last_update_utc': 'Wed, 27 May 2026 00:00:01 +0000',
 'time_next_update_unix': 1779926401,
 'time_next_update_utc': 'Thu, 28 May 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.77}

In [127]:
# Second tool for conversion

In [128]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [129]:
@tool
def convert(base_currency_value:int,conversion_rate:Annotated[float,InjectedToolArg])->float:
  """Given a currency conversion rate this function calculates the target currency value from a given base currency value"""
  return base_currency_value*conversion_rate

# Annotated[float,InjectedToolArg] tells that when llm calling this tool it will not set the converion rate. Developer will inject this value after running earlier tools

In [130]:
convert.invoke({'base_currency_value':10,'conversion_rate': 95.77})

957.6999999999999

In [131]:
# creating llm with tools - tool binding

In [132]:
llm_with_tools=llm.bind_tools([get_conversion_factor,convert])

In [133]:
# tool calling

In [151]:
query=HumanMessage("What is the conversion factor between USD and INR , based on that can you convert 10 USD to INR")

In [152]:
messages=[query]

In [153]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR , based on that can you convert 10 USD to INR', additional_kwargs={}, response_metadata={})]

In [154]:
response_from_llm=llm_with_tools.invoke(messages)

In [155]:
response_from_llm

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 108, 'total_tokens': 130, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_944bbf963c', 'id': 'chatcmpl-Dk1JL7kexwURWdJ21KujN9SNyFmoQ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e67f3-ebdd-7da0-8df7-8da535f3d4d8-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_muQOD785aluu1ZHdwkirVLBK', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 108, 'output_tokens': 22, 'total_tokens': 130, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details':

In [156]:
messages.append(response_from_llm)

In [157]:
response_from_llm.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_muQOD785aluu1ZHdwkirVLBK',
  'type': 'tool_call'}]

In [158]:
for tool_call in response_from_llm.tool_calls:
  print(tool_call)

{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_muQOD785aluu1ZHdwkirVLBK', 'type': 'tool_call'}


In [159]:
import json
for tool_call in response_from_llm.tool_calls:
  # Execute the first tool and get value of conversion rate
  if tool_call['name']=='get_conversion_factor':
    tool_message_one=get_conversion_factor.invoke(tool_call)
    conversion_rate=(json.loads(tool_message_one.content)['conversion_rate'])
    messages.append(tool_message_one)
    # Execute the 2nd tool using the conversion rate from first tool
  if tool_call['name']=='convert':
    tool_call['args']['conversion_rate']=conversion_rate
    tool_message_two=convert.invoke(tool_call)
    messages.append(tool_message_two)



In [160]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR , based on that can you convert 10 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 108, 'total_tokens': 130, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_944bbf963c', 'id': 'chatcmpl-Dk1JL7kexwURWdJ21KujN9SNyFmoQ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e67f3-ebdd-7da0-8df7-8da535f3d4d8-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_muQOD785aluu1ZHdwkirVLBK', 'type': 'tool_call'}], invalid_tool

In [161]:
result=llm_with_tools.invoke(messages)

In [162]:
result

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 278, 'total_tokens': 293, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_944bbf963c', 'id': 'chatcmpl-Dk1JXrWSdcxyXrKsUwd0wGfY2XgF0', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e67f4-169f-7f53-a566-0a4fe59e29ea-0', tool_calls=[{'name': 'convert', 'args': {'base_currency_value': 10}, 'id': 'call_TH3xYUnxnF51V7m28MOtMzbU', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 278, 'output_tokens': 15, 'total_tokens': 293, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [163]:
messages.append(result)

In [164]:
import json
for tool_call in result.tool_calls:
  # Execute the first tool and get value of conversion rate
  if tool_call['name']=='convert':
    tool_call['args']['conversion_rate']=conversion_rate
    tool_message_two=convert.invoke(tool_call)
    print(tool_message_two)
    messages.append(tool_message_two)


content='957.6999999999999' name='convert' tool_call_id='call_TH3xYUnxnF51V7m28MOtMzbU'


In [165]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR , based on that can you convert 10 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 108, 'total_tokens': 130, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_944bbf963c', 'id': 'chatcmpl-Dk1JL7kexwURWdJ21KujN9SNyFmoQ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e67f3-ebdd-7da0-8df7-8da535f3d4d8-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_muQOD785aluu1ZHdwkirVLBK', 'type': 'tool_call'}], invalid_tool

In [166]:
result_final=llm_with_tools.invoke(messages)

In [167]:
result_final.content

'The conversion factor between USD and INR is 95.77. Therefore, converting 10 USD to INR results in approximately 957.70 INR.'